In [1]:
import pandas as pd

train_df = pd.read_csv("../data/ddi_train.csv")
test_df = pd.read_csv("../data/ddi_test.csv")

label2id = {
    "NO-REL": 0,
    "effect": 1,
    "mechanism": 2,
    "advise": 3,
    "int": 4
}

id2label = {v: k for k, v in label2id.items()}

print("Train:", len(train_df))
print("Test:", len(test_df))
print(train_df.head(3))

Train: 27792
Test: 6657
                                            sentence          entity1  \
0  Differential regulation of tyrosine phosphoryl...  contortrostatin   
1  Differential regulation of tyrosine phosphoryl...  contortrostatin   
2  Differential regulation of tyrosine phosphoryl...       echistatin   

      entity2 relation  
0  echistatin   NO-REL  
1  flavoridin   NO-REL  
2  flavoridin   NO-REL  


In [2]:
# Before entity markers
sentence = "Warfarin increases the anticoagulant effect of Aspirin."
e1 = "Warfarin"
e2 = "Aspirin"

print("=== WITHOUT entity markers ===")
print(sentence)
print()

# After entity markers
def add_entity_markers(sentence, e1, e2):
    # Replace entity mentions with marked versions
    # Important: replace longer strings first to avoid partial replacements
    entities = sorted([(e1, "[E1]", "[/E1]"), (e2, "[E2]", "[/E2]")], 
                      key=lambda x: len(x[0]), reverse=True)
    
    marked = sentence
    for entity, start_tag, end_tag in entities:
        marked = marked.replace(entity, f"{start_tag}{entity}{end_tag}", 1)
    
    return marked

marked = add_entity_markers(sentence, e1, e2)
print("=== WITH entity markers ===")
print(marked)
print()
print("This tells BERT exactly which two entities to focus on")

=== WITHOUT entity markers ===
Warfarin increases the anticoagulant effect of Aspirin.

=== WITH entity markers ===
[E1]Warfarin[/E1] increases the anticoagulant effect of [E2]Aspirin[/E2].

This tells BERT exactly which two entities to focus on


In [3]:
def prepare_re_sample(row):
    sentence = str(row["sentence"])
    e1 = str(row["entity1"])
    e2 = str(row["entity2"])
    
    # Add entity markers
    marked_sentence = add_entity_markers(sentence, e1, e2)
    
    return marked_sentence

# Apply to train and test
train_df["marked_sentence"] = train_df.apply(prepare_re_sample, axis=1)
test_df["marked_sentence"] = test_df.apply(prepare_re_sample, axis=1)

# Add numeric labels
train_df["label"] = train_df["relation"].map(label2id)
test_df["label"] = test_df["relation"].map(label2id)

print("=== Sample marked sentences ===\n")
for i, row in train_df[train_df["relation"] != "NO-REL"].head(5).iterrows():
    print(f"Relation: {row['relation']}")
    print(f"Marked:   {row['marked_sentence'][:150]}")
    print()

=== Sample marked sentences ===

Relation: effect
Marked:   [E1]Echistatin[/E1] alone had no effect on tyrosine phosphorylation in T24 cells, but dose-dependently inhibits the effects of [E2]contortrostatin[/E2

Relation: effect
Marked:   [E1]Flavoridin[/E1] alone was found to have no effect on CAS, but can completely block [E2]contortrostatin[/E2]-induced phosphorylation of this protei

Relation: effect
Marked:   Prior administration of [E1]4-methylpyrazole[/E1] (90 mg kg(-1) body weight) was shown to prevent the conversion of [E2]1,3-difluoro-2-propanol[/E2] (

Relation: effect
Marked:   We conclude that the prophylactic and antidotal properties of [E1]4-methylpyrazole[/E1] seen in animals treated with [E2]1,3-difluoro-2-propanol[/E2] 

Relation: effect
Marked:   Intrathecal injection of [E1]naloxone[/E1] at doses of 0.4 to 40 micrograms caused a dose-related blockade of the inhibition of the tail-flick respons



In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

# Add entity marker tokens to vocabulary
special_tokens = ["[E1]", "[/E1]", "[E2]", "[/E2]"]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

print("Vocabulary size before:", 28996)
print("Vocabulary size after: ", len(tokenizer))
print("Added tokens:", special_tokens)

# Verify tokens are recognized
test_text = "[E1]Warfarin[/E1] increases effect of [E2]Aspirin[/E2]"
tokens = tokenizer.tokenize(test_text)
print("\nTokenized with markers:", tokens)

/Users/angelinagupta/biomedical-nlp-project/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Vocabulary size before: 28996
Vocabulary size after:  29000
Added tokens: ['[E1]', '[/E1]', '[E2]', '[/E2]']

Tokenized with markers: ['[E1]', 'war', '##fari', '##n', '[/E1]', 'increases', 'effect', 'of', '[E2]', 'as', '##pi', '##rin', '[/E2]']


In [5]:
def tokenize_re_dataset(texts, labels, tokenizer, max_length=256):
    encodings = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encodings, labels

import torch
from torch.utils.data import Dataset

class DDIDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Create datasets
train_encodings, train_labels = tokenize_re_dataset(
    train_df["marked_sentence"].tolist(),
    train_df["label"].tolist(),
    tokenizer
)

test_encodings, test_labels = tokenize_re_dataset(
    test_df["marked_sentence"].tolist(),
    test_df["label"].tolist(),
    tokenizer
)

train_dataset = DDIDataset(train_encodings, train_labels)
test_dataset = DDIDataset(test_encodings, test_labels)

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))
print("Sample keys:", train_dataset[0].keys())

Train dataset size: 27792
Test dataset size: 6657
Sample keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


In [6]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels_array = np.array(train_df["label"].tolist())
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels_array),
    y=labels_array
)

print("=== Class Weights ===\n")
for idx, weight in enumerate(class_weights):
    print(f"{id2label[idx]:12} → weight: {weight:.4f}")

print("\nHigher weight = model penalized more for missing this class")
print("This handles the 85% NO-REL imbalance")

=== Class Weights ===

NO-REL       → weight: 0.2338
effect       → weight: 3.2948
mechanism    → weight: 4.2141
advise       → weight: 6.7293
int          → weight: 29.5660

Higher weight = model penalized more for missing this class
This handles the 85% NO-REL imbalance


In [7]:
# Save marked sentences and labels to CSV for Colab
train_df[["marked_sentence", "relation", "label"]].to_csv("../data/ddi_train_marked.csv", index=False)
test_df[["marked_sentence", "relation", "label"]].to_csv("../data/ddi_test_marked.csv", index=False)

# Save class weights
import json
weights_dict = {id2label[i]: float(class_weights[i]) for i in range(len(class_weights))}
with open("../data/class_weights.json", "w") as f:
    json.dump(weights_dict, f, indent=2)

print("Saved:")
print("  data/ddi_train_marked.csv")
print("  data/ddi_test_marked.csv")
print("  data/class_weights.json")

Saved:
  data/ddi_train_marked.csv
  data/ddi_test_marked.csv
  data/class_weights.json


In [8]:
import pandas as pd

df = pd.read_csv("../data/ddi_train_marked.csv")
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print("\nSample:")
print(df.head(3))

Columns: ['marked_sentence', 'relation', 'label']
Shape: (27792, 3)

Sample:
                                     marked_sentence relation  label
0  Differential regulation of tyrosine phosphoryl...   NO-REL      0
1  Differential regulation of tyrosine phosphoryl...   NO-REL      0
2  Differential regulation of tyrosine phosphoryl...   NO-REL      0
